# Kraken fine-tuning - raport eksperymentu

Ten notebook porz?dkuje lokalny eksperyment fine-tuningu Kraken na polskim datasecie pisma odr?cznego oraz ocen? generalizacji na Malaysian. Wyniki i wykresy s? zapisane obok notebooka w folderach `tables/` i `figures/`.

## Legenda

- `Zero-shot McCATMuS` - bazowy model Kraken/McCATMuS bez fine-tuningu.
- `Kraken standard` - fine-tuning 10 epok na polskim zbiorze treningowym.
- `Kraken oversampling` - fine-tuning 10 epok na wariancie treningowym z powieleniem pr?bek dysgrafii.
- `native` - oryginalny preprocessing polskiego datasetu.
- `raw` - Malaysian bez odwr?cenia kolor?w.
- `auto_invert` - Malaysian po automatycznym odwr?ceniu czarnego t?a i bia?ych liter.
- `corpus_CER/WER` - metryka liczona globalnie po zsumowaniu odleg?o?ci edycyjnej i d?ugo?ci referencji.
- `CLA/CRW` - character-level accuracy i correctly recognized words wed?ug wzor?w u?ywanych w pracy.

## Metodologia

Trening wykonano lokalnie na Windows/GPU przez `ketos train`. Dane liniowe przekonwertowano do PAGE XML z jedn? lini? tekstu na obraz. W ewaluacji u?yto sztucznej baseline przechodz?cej przez ?rodek obrazu, poniewa? fine-tunowany model Kraken ma `seg_type=baselines`.

In [ ]:
from pathlib import Path
import pandas as pd
REPORT = Path(r'C:/Magisterka/Kraken_final_report')
TABLES = REPORT / 'tables'
FIGURES = REPORT / 'figures'
comparison = pd.read_csv(TABLES / 'kraken_final_comparison_summary.csv')
comparison

## Krzywa treningu

Walidacyjna dok?adno?? raportowana przez `ketos` ros?a szybko i po 10 epokach osi?gn??a bardzo podobny poziom dla obu wariant?w.

![Kraken validation accuracy](figures/kraken_training_val_accuracy.png)

## Polski test - CER/WER

Fine-tuning radykalnie poprawia Kraken wzgl?dem zero-shot. Oversampling daje ma??, ale sp?jn? popraw? na ca?ym te?cie.

![Kraken Polish CER WER](figures/kraken_polish_cer_wer.png)

In [ ]:
pd.read_csv(TABLES / 'kraken_polish_test_summary.csv')[[ 'run_label', 'samples', 'corpus_cer', 'corpus_wer', 'corpus_cla', 'corpus_crw', 'inference_seconds_mean' ]]

## Grupy trudno?ci

Najwa?niejszy efekt oversamplingu wida? w grupie `dysgrafia`: spada zar?wno CER, jak i WER.

![Kraken Polish group CER](figures/kraken_polish_group_cer.png)

In [ ]:
pd.read_csv(TABLES / 'kraken_polish_test_group_summary.csv')[[ 'run_label', 'group', 'samples', 'corpus_cer', 'corpus_wer', 'corpus_cla', 'corpus_crw' ]]

## Malaysian - generalizacja zewn?trzna

Po fine-tuningu na polskich danych Kraken silnie specjalizuje si? w nowej domenie. Wyniki na Malaysian pozostaj? s?abe, zw?aszcza bez odwr?cenia kolor?w. `auto_invert` nadal pomaga, ale nie rozwi?zuje problemu j?zyka/domeny.

![Kraken Malaysian CER WER](figures/kraken_malaysian_cer_wer.png)

In [ ]:
pd.read_csv(TABLES / 'kraken_malaysian_external_summary.csv')[[ 'run_label', 'dataset', 'preprocessing_variant', 'samples', 'corpus_cer', 'corpus_wer', 'corpus_cla', 'corpus_crw', 'inference_seconds_mean' ]]

## Czas inferencji

Kraken jest szybki jako klasyczny model HTR. W tym notebooku raportowany jest ?redni czas na lini? oraz RSS procesu; `cuda_peak_*` nie by?o dost?pne dla lokalnego evaluatora Kraken.

![Kraken inference time](figures/kraken_inference_time.png)

## Wnioski do pracy

- Fine-tuning Kraken ma sens dla polskiego datasetu: CER spada z poziomu zero-shot oko?o 0.77 do oko?o 0.023-0.025.
- Oversampling dysgrafii poprawia wyniki w grupie dysgrafia, ale nie zmienia radykalnie ca?ej walidacji.
- Kraken jest szybki i klasyczny metodologicznie, ale mniej odporny domenowo ni? Qwen.
- Wyniki Malaysian pokazuj?, ?e po fine-tuningu na polskich danych model traci u?yteczno?? jako model zewn?trznie generalizuj?cy.